[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaellopesdesa/ML_Mandi/blob/main/activities/application/Exercise_2_3_parameter_fitting.ipynb)

# Exercise 2.3 — Parameter fitting

HSF-India lectures at IIT Mandi. Adapted from the [ML4HEP-TIFR NSBI tutorial](https://github.com/iris-hep/nsbi-lhc-toolkit/tree/b1d3cb8c704ee7a54c99367cf6578e5244ac2671/workshops/ml4hep_tifr_colab).

Run the setup cell first, then run the cells in order. Use a **CPU runtime**.
Everything is generated on the runtime's local disk: **20,000 signal and 20,000 background events**, with seed 42.
Training uses two hidden layers of 64 neurons and at most 20 epochs. The expected physics yields are unchanged, so the smaller simulated samples give noisier estimates.
The original exercises and worked likelihood implementations are retained.

A fresh runtime automatically runs the small trainings from 2.2a and 2.2b if their models are missing. To fit your own modified networks, first upload the ZIP files produced by those notebooks using Colab's **Files** sidebar, then run this notebook. The notebook checks that the models used the same data.


In [ ]:
# Run first: local storage only, in Colab or local Jupyter.
import os
import sys
from pathlib import Path
from urllib.request import urlretrieve

REPOSITORY = "rafaellopesdesa/ML_Mandi"
REF = "main"
N_BKG = N_SIG = 20_000
SEED = 42

if "google.colab" in sys.modules:
    WORKDIR = Path("/content/ML_Mandi_application")
elif (Path.cwd() / "setup_local.py").is_file():
    WORKDIR = Path.cwd()
elif (Path.cwd() / "activities/application/setup_local.py").is_file():
    WORKDIR = Path.cwd() / "activities/application"
else:
    WORKDIR = Path.cwd() / "ML_Mandi_application"

WORKDIR = WORKDIR.resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
helper = WORKDIR / "setup_local.py"
if not helper.is_file():
    urlretrieve(
        f"https://raw.githubusercontent.com/{REPOSITORY}/{REF}/activities/application/setup_local.py",
        helper,
    )
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

from setup_local import setup
setup(WORKDIR, repository=REPOSITORY, ref=REF, n_bkg=N_BKG, n_sig=N_SIG, seed=SEED)


Parameter fitting
---

Now that we have the trained density ratios, it is time to build the full statistical model

<!-- $$ \frac{L(\mu)}{\prod_e p_\mathrm{ref}(x_e)} = \prod_{x_e\in \{x\}} \frac{1}{\mu \cdot \lambda_\mathrm{sig} + \lambda_\mathrm{bkg}} \bigg[\mu \cdot \lambda_\mathrm{sig} \cdot \underbrace{\frac{p_\mathrm{sig}(x_e)}{p_\mathrm{ref}(x_e)}}_\text{trained model} + \lambda_\mathrm{bkg} \cdot \underbrace{\frac{p_\mathrm{bkg}(x_e)}{p_\mathrm{ref}(x_e)}}_\text{trained model}\bigg]$$ -->

$$-2 \ln \frac{L(\mu)}{\prod_e p_\mathrm{ref}(x_e)} = -2 \sum_{x_e\in \{x\}} \ln \bigg(\frac{1}{\mu \cdot \lambda_\mathrm{sig} + \lambda_\mathrm{bkg}} \bigg[\mu \cdot \lambda_\mathrm{sig} \cdot \underbrace{\frac{p_\mathrm{sig}(x_e)}{p_\mathrm{ref}(x_e)}}_\text{trained model} + \lambda_\mathrm{bkg} \cdot \underbrace{\frac{p_\mathrm{bkg}(x_e)}{p_\mathrm{ref}(x_e)}}_\text{trained model}\bigg]\bigg)$$

which can be used for the final test statistic:

$$ t_\mu = -2 \big[ \ln L(\mu) - \ln L(\hat{\mu}) \big]$$

where the reference terms have canceled out. 

In this notebook, we will use the `nsbi-common-utils` to build the statistical model and perform fits with the test statistic.  


In [ ]:
import os
import nsbi_common_utils
import pandas as pd
import numpy as np
import mplhep as hep
import pickle
import matplotlib.pyplot as plt
from nsbi_common_utils.training.utils import load_trained_model


The setup cell has generated the samples locally. Keep this relative path.

In [ ]:
BASE_PATH = "./dataframes/"

Load the signal and background datasets

In [ ]:
signal = pd.read_parquet(f"{BASE_PATH}/signal.parquet")
background = pd.read_parquet(f"{BASE_PATH}/background.parquet")

### Train / inference split

The density ratios must be **trained** and the likelihood **evaluated** on *different* events, otherwise the parameter fit inherits the classifier's overfitting. We split each sample with a fixed seed: the training notebooks (2a, 2b) keep the *training* half and the fitting notebook (2.3) keeps the *inference* half.

`SPLIT_SEED` and `TRAIN_FRACTION` **must be identical across all three notebooks** — the shared seed is what guarantees the two halves are disjoint. Change `TRAIN_FRACTION` (or pass `n_train=<int>` to `split_train_inference`) to adjust how many events go to training vs. inference.

In [ ]:
from utils import split_train_inference

SPLIT_SEED = 0
TRAIN_FRACTION = 0.5  # fraction of each sample used for TRAINING

_, signal = split_train_inference(signal, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED)
_, background = split_train_inference(background, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED)

print(f"Fitting on {len(signal):,} signal + {len(background):,} background events")

Get the total yields for the estimation of $\lambda_\mathrm{sig}, \lambda_\mathrm{bkg}$ to be used in the statistical model

In [ ]:
TOTAL_YIELD = {}
TOTAL_YIELD["signal"] = signal.weight.sum()
TOTAL_YIELD["background"] = background.weight.sum()

Since we are using a weighted dataset, we also need a weighted sum of log-likelihoods and thus need to extract the weight vector for the data we are fitting

$$-2 \ln L(\mu) = -2 \sum_{x_e\in \{x\}} w_e \cdot \ln \bigg(\frac{1}{\mu \cdot \lambda_\mathrm{sig} + \lambda_\mathrm{bkg}} \bigg[\mu \cdot \lambda_\mathrm{sig} \cdot \frac{p_\mathrm{sig}(x_e)}{p_\mathrm{ref}(x_e)} + \lambda_\mathrm{bkg} \cdot \frac{p_\mathrm{bkg}(x_e)}{p_\mathrm{ref}(x_e)}\bigg]\bigg)$$

In [ ]:
asimov_dataset = pd.concat([background, signal], ignore_index=True).astype('float32').copy()
weights_asimov = np.array(asimov_dataset.weight)
PATH_TO_WEIGHTS = "saved_weights/asimov_weight_array.npy"
os.makedirs(os.path.dirname(PATH_TO_WEIGHTS), exist_ok=True)
np.save(PATH_TO_WEIGHTS, weights_asimov)

Features used in the training setup

In [ ]:
training_features = ["x1", "x2", "x3", "x4", "x5"]

Set the path to where the estimated density ratios from trained models will be saved, to be later used in the fits

In [ ]:
PATH_TO_RATIOS = {}
PATH_TO_RATIOS["signal"] = "saved_ratios_asimov/ratio_SigvsRef.npy"
PATH_TO_RATIOS["background"] = "saved_ratios_asimov/ratio_BkgvsRef.npy"

In [ ]:
for path in PATH_TO_RATIOS.values():
    os.makedirs(os.path.dirname(path), exist_ok=True)

Load the trained models, and evaluate on data. For this example, we use the Asimov data only.

A simple call to `nsbi_common_utils.training.utils.predict_with_model` with the path to the saved neural networks is enough to get the predictions.

The networks were trained with the binary cross-entropy loss, so their raw output is the classifier score $\hat{s}(x)$. The density ratio we want is obtained via the monotonic transformation

$$ \hat{r}(x) = \frac{\hat{s}(x)}{1.0 - \hat{s}(x)} $$

**`predict_with_model` applies this transformation for you** and returns the density ratio directly — so do *not* convert its output again (doing so would compute $r/(1-r)$, which is negative wherever $r>1$ and diverges at $r\approx 1$, producing `nan` in the fit).


In [ ]:
# Reuse uploaded/local networks, or reproduce 2.2a and 2.2b in this runtime.
from setup_local import ensure_trained_models
ensure_trained_models()


In [ ]:
import pickle   # add to the notebook's import cell if not already there

ensemble_size = 1
USE_CALIBRATION = False          # load & apply the saved histogram calibrator

In [ ]:
ensemble_size = 1
ratios = []
for ens_index in range(ensemble_size):
    path_to_saved_scaler = f"models_SigvsRef/model_scaler{ens_index}.bin"
    path_to_saved_model  = f"models_SigvsRef/model{ens_index}.onnx"
    path_to_calibrator   = f"models_SigvsRef/model_calibrated_hist{ens_index}.obj"

    scaler, model = load_trained_model(path_to_saved_model, path_to_saved_scaler)

    # histogram calibrator saved during training (HistogramCalibrator, pickled)
    calibration_model = None
    if USE_CALIBRATION and os.path.exists(path_to_calibrator):
        with open(path_to_calibrator, "rb") as f:
            calibration_model = pickle.load(f)

    ratio = nsbi_common_utils.training.utils.predict_with_model(
        asimov_dataset[training_features],
        scaler=scaler,
        model=model,
        calibration_model=calibration_model,   # None -> raw ratio, as before
    )
    ratios.append(ratio)

prediction_ratio = np.mean(np.stack(ratios, axis=0), axis=0)
np.save(PATH_TO_RATIOS["signal"], prediction_ratio)


In [ ]:
ratios = []
for ens_index in range(ensemble_size):
    path_to_saved_scaler = f"models_BkgvsRef/model_scaler{ens_index}.bin"
    path_to_saved_model  = f"models_BkgvsRef/model{ens_index}.onnx"
    path_to_calibrator   = f"models_BkgvsRef/model_calibrated_hist{ens_index}.obj"

    scaler, model = load_trained_model(path_to_saved_model, path_to_saved_scaler)

    # histogram calibrator saved during training (HistogramCalibrator, pickled)
    calibration_model = None
    if USE_CALIBRATION and os.path.exists(path_to_calibrator):
        with open(path_to_calibrator, "rb") as f:
            calibration_model = pickle.load(f)

    ratio = nsbi_common_utils.training.utils.predict_with_model(
        asimov_dataset[training_features],
        scaler=scaler,
        model=model,
        calibration_model=calibration_model,   # None -> raw ratio, as before
    )
    ratios.append(ratio)

prediction_ratio = np.mean(np.stack(ratios, axis=0), axis=0)
np.save(PATH_TO_RATIOS["background"], prediction_ratio)


Build a workspace
---

Now its time to build a workspace object, which is a serialization of the statistical model, that can be used to perform parameter fits. 

The following JSON serialization standard has been heavily inspired by the `pyhf` framework by design - allowing future avenues of interoperability.

In [ ]:
def make_workspace():
    """Toy workspace for this example."""
    return {
        "channels": [{
            "name": "SR",
            "type": "unbinned",
            "weights": PATH_TO_WEIGHTS,
            "samples": [
                {"name": "signal",
                 "data": [TOTAL_YIELD["signal"]],
                 "ratios": PATH_TO_RATIOS["signal"],
                 "modifiers": [{"name": "mu", "type": "normfactor"}]},
                {"name": "background",
                 "data": [TOTAL_YIELD["background"]],
                 "ratios": PATH_TO_RATIOS["background"],
                 "modifiers": []},
            ],
        }],
        # "observations": [{"name": "SR", "data": DATA}],
        "measurements": [{
            "name": "meas",
            "config": {
                "poi": "mu",
                "parameters": [
                    {"bounds": [[0, 10]], "inits": [1], "name": "mu"},
                    {"bounds": [[0, 10]], "inits": [1], "name": "mu_bkg"},
                ],
            },
        }],
        "version": "1.0.0",
    }


In [ ]:
ws_nsbi = make_workspace()

Pass the workspace to the model API `nsbi_common_utils.models.sbi_parametric_model`, which internally builds the JAX-based model to be used for fitting 

In [ ]:
model_nsbi = nsbi_common_utils.models.sbi_parametric_model(workspace=ws_nsbi, 
                                                            measurement_to_fit="meas")

Get the hyperparameters

In [ ]:
list_params, init_values = model_nsbi.get_model_parameters()
num_unconstrained = model_nsbi.num_unconstrained_param

Pass the model to an inference library - currently using an "in-house" simple API `nsbi_common_utils.inference`, but the longer term goal is that this model can be used with standard fitting frameworks.

In [ ]:
inf_nsbi = nsbi_common_utils.inference.inference(
            model_nll=model_nsbi.model,
            initial_values=init_values,
            list_parameters=list_params,
            num_unconstrained_params=num_unconstrained,
            model_grad=model_nsbi.model_grad
        )

Get the global fit results

In [ ]:
print("\n" + "="*40)
print(" NSBI FIT RESULTS ")
print("="*40 + "\n")
inf_nsbi.perform_fit(freeze_params=[])

Build the $t_\mu$ curve for parameter estimation

In [ ]:
scan_points_nsbi, NLL_value_nsbi, scan_points_StatOnly_nsbi, NLL_value_StatOnly_nsbi = inf_nsbi.perform_profile_scan(parameter_name = 'mu', 
                             freeze_params = [], 
                             bound_range = (-1.0, 3.0), 
                             fit_strategy = 0, 
                             doStatOnly = True,
                             size=31)

In [ ]:
plt.plot(scan_points_nsbi, NLL_value_nsbi)
plt.axis(ymin=0)
plt.xlabel(r"$\mu_\mathrm{signal}$")
plt.ylabel(r"$t_\mu$")

Seeing a bias in the estimator? This is expected in any per-event approach due to the MC noise from limited datasets and sub-optimal Neural Network training. 

Try building better architectures, increasing the dataset size for training, building ensembles of Neural Networks, generating more data for asimov using instructions on the README, or train larger ensembles.

Let's compare with the truth density ratios and see if we reproduce the optimal sensitivity.

In [ ]:
from utils import background_components, signal_components, mixture_density, smearing_parameters
from scipy.special import logsumexp
from scipy.stats import multivariate_normal

def reco_components_from_truth_components(components):
    scale, resolution = smearing_parameters()
    scale = np.asarray(scale, dtype=float)
    resolution = np.asarray(resolution, dtype=float)
    D = np.diag(scale)
    response_cov = np.diag(resolution ** 2)

    reco_components = []
    for frac, mean_z, cov_z in components:
        mean_x = scale * np.asarray(mean_z, dtype=float)
        cov_x = D @ np.asarray(cov_z, dtype=float) @ D.T + response_cov
        reco_components.append((frac, mean_x, cov_x))
    return reco_components


def mixture_log_density(x, components):
    x = np.asarray(x, dtype=float)
    fracs = np.asarray([c[0] for c in components], dtype=float)
    fracs = fracs / fracs.sum()

    terms = []
    for f, (_, mean, cov) in zip(fracs, components):
        terms.append(
            np.log(f) + multivariate_normal(mean=mean, cov=cov, allow_singular=False).logpdf(x)
        )
    return logsumexp(np.vstack(terms), axis=0)


TRUTH_RECO_COMPONENTS = {
    "background": reco_components_from_truth_components(background_components()),
    "signal": reco_components_from_truth_components(signal_components()),
}

# Analytic (truth) density ratios, using the same reference as the training:
# p_ref = (lam_bkg p_bkg + lam_sig p_sig) / (lam_bkg + lam_sig)
lam_bkg = TOTAL_YIELD["background"]
lam_sig = TOTAL_YIELD["signal"]
X_asimov = asimov_dataset[training_features].values
p_bkg = np.exp(mixture_log_density(X_asimov, TRUTH_RECO_COMPONENTS['background']))
p_sig = np.exp(mixture_log_density(X_asimov, TRUTH_RECO_COMPONENTS['signal']))
p_ref = (lam_bkg * p_bkg + lam_sig * p_sig) / (lam_bkg + lam_sig)

PATH_TO_RATIOS_TRUTH = {
    "signal": "saved_ratios_asimov/ratio_SigvsRef_truth.npy",
    "background": "saved_ratios_asimov/ratio_BkgvsRef_truth.npy",
}
np.save(PATH_TO_RATIOS_TRUTH["signal"], p_sig / p_ref)
np.save(PATH_TO_RATIOS_TRUTH["background"], p_bkg / p_ref)

In [ ]:
def make_workspace_truth():
    """Same workspace as make_workspace(), but with the analytic (truth) ratios."""
    ws = make_workspace()
    ws["channels"][0]["samples"][0]["ratios"] = PATH_TO_RATIOS_TRUTH["signal"]
    ws["channels"][0]["samples"][1]["ratios"] = PATH_TO_RATIOS_TRUTH["background"]
    return ws


model_truth = nsbi_common_utils.models.sbi_parametric_model(
    workspace=make_workspace_truth(), measurement_to_fit="meas")
list_params_t, init_values_t = model_truth.get_model_parameters()
inf_truth = nsbi_common_utils.inference.inference(
    model_nll=model_truth.model,
    initial_values=init_values_t,
    list_parameters=list_params_t,
    num_unconstrained_params=model_truth.num_unconstrained_param,
    model_grad=model_truth.model_grad,
)

SCAN_RANGE = (-1.0, 3.0)
scan_truth, tmu_truth = inf_truth.perform_profile_scan(
    parameter_name="mu", freeze_params=[], bound_range=SCAN_RANGE, fit_strategy=0, size=31)
scan_pred, tmu_pred = inf_nsbi.perform_profile_scan(
    parameter_name="mu", freeze_params=[], bound_range=SCAN_RANGE, fit_strategy=0, size=31)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(scan_pred, tmu_pred, lw=2, label="Approximated (NN) ratios")
ax.plot(scan_truth, tmu_truth, lw=2, ls="--", label="Truth ratios")
ax.axvline(1.0, color="black", ls=":", lw=1, alpha=0.6)
ax.set_ylim(bottom=0)
ax.set_xlabel(r"$\mu_\mathrm{signal}$")
ax.set_ylabel(r"$t_\mu$")
ax.legend()

Exercise: build the likelihood by hand
---

`sbi_parametric_model` built the NLL for us above. To see what it is doing under the hood, let's construct the likelihood **manually** and pass it straight to `nsbi_common_utils.inference.inference`.

The model is an extended likelihood: a Poisson term for the total rate times the per-event probability density,

$$\frac{L(\mu)}{\prod_{x_e\in\{x\}} p_\mathrm{ref}(x_e)} = \underbrace{\mathrm{Pois}\big(N_\mathrm{tot} \mid \mu\lambda_\mathrm{sig}+\lambda_\mathrm{bkg}\big)}_{\text{extended Poisson term}} \; \prod_{x_e\in\{x\}} \underbrace{\frac{\mu \lambda_\mathrm{sig} \frac{p_\mathrm{sig}(x_e)}{p_\mathrm{ref}(x_e)} + \lambda_\mathrm{bkg} \frac{p_\mathrm{bkg}(x_e)}{p_\mathrm{ref}(x_e)}}{\mu \lambda_\mathrm{sig} + \lambda_\mathrm{bkg}}}_{\text{probability density}}$$

where $N_\mathrm{tot}=\sum_e w_e$ is the observed total yield. The `inference` engine minimises $-2\ln L(\mu)$, which you implement below (writing it in JAX gives the gradient for free via `jax.grad`).

Everything you need is already computed earlier in this notebook:

| symbol | variable |
|---|---|
| $p_\mathrm{sig}/p_\mathrm{ref}$ | `r_sig` (from `PATH_TO_RATIOS["signal"]`) |
| $p_\mathrm{bkg}/p_\mathrm{ref}$ | `r_bkg` (from `PATH_TO_RATIOS["background"]`) |
| $w_e$ | `weights` (from `PATH_TO_WEIGHTS`) |
| $\lambda_\mathrm{sig}, \lambda_\mathrm{bkg}$ | `lam_sig`, `lam_bkg` (from `TOTAL_YIELD`) |
| $N_\mathrm{tot}$ | `N_total` ($=\sum_e w_e$) |

Fill in `nll_manual` below, then build an `inference` object and fit.


In [ ]:
import jax
import jax.numpy as jnp

# Ingredients of the likelihood, evaluated on the (inference-split) Asimov data
r_sig = jnp.asarray(np.load(PATH_TO_RATIOS["signal"]))
r_bkg = jnp.asarray(np.load(PATH_TO_RATIOS["background"]))
weights = jnp.asarray(np.load(PATH_TO_WEIGHTS))
lam_sig = float(TOTAL_YIELD["signal"])
lam_bkg = float(TOTAL_YIELD["background"])
N_total = float(weights.sum())   # observed total yield


def nll_manual(params):
    mu = params[0]
    nu_tot = mu * lam_sig + lam_bkg
    per_event = (mu * lam_sig * r_sig + lam_bkg * r_bkg) / nu_tot
    return -2.0 * jnp.sum(weights * jnp.log(per_event)) - 2.0 * (N_total * jnp.log(nu_tot) - nu_tot)


inf_manual = nsbi_common_utils.inference.inference(
    model_nll=jax.jit(nll_manual), initial_values=[1.0],
    list_parameters=["mu"], num_unconstrained_params=1,
    model_grad=jax.jit(jax.grad(nll_manual)),
)
inf_manual.perform_fit(freeze_params=[])
